In [ ]:

# ===== D_Cortex comprehensive A100 validation notebook (single cell) =====
# This cell:
#   1) configures Colab/A100 for bf16 + TF32 + Flash SDPA
#   2) clones/fixes the repo layout if needed
#   3) patches attention modules to use Flash Attention kernels through PyTorch SDPA
#   4) tokenizes TinyStories into memmap binaries
#   5) builds a synthetic memory curriculum
#   6) trains D_Cortex end-to-end with a complete loop
#   7) runs comprehensive evaluation, ablation, checkpoints, and plots
#
# Designed for Colab A100. No GradScaler is used: bf16 is native on A100.

import os, sys, math, time, json, shutil, random, tempfile, subprocess, importlib, textwrap
from pathlib import Path
from dataclasses import asdict
from typing import Dict, List, Tuple

def _ensure(pkg, pip_name=None):
    pip_name = pip_name or pkg
    try:
        importlib.import_module(pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for pkg, pip_name in [
    ("numpy", None),
    ("matplotlib", None),
    ("datasets", None),
    ("transformers", None),
    ("tqdm", None),
]:
    _ensure(pkg, pip_name)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from datasets import load_dataset
from transformers import GPT2TokenizerFast
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# -----------------------
# User-tunable validation config
# -----------------------
RUN_NAME               = "dcortex_a100_validation"
REPO_URL               = "https://github.com/NEURALMORPHIC-FIELDS/D_Cortex"
WORKDIR                = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR               = WORKDIR / "D_Cortex"
OUT_DIR                = WORKDIR / "dcortex_runs" / RUN_NAME
DATA_DIR               = WORKDIR / "dcortex_data"
CKPT_DIR               = OUT_DIR / "checkpoints"
REPORT_DIR             = OUT_DIR / "report"
for d in [OUT_DIR, DATA_DIR, CKPT_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED                   = 1337
TRAIN_STORIES          = 200_000       # enough for strong signal, not full pretraining
VAL_STORIES            = 5_000
SEQ_LEN                = 1024          # set to 2048 if you want the full default context stress
MICRO_BSZ              = 16            # A100-safe for this model at 1024 with SDPA/flash kernels
GRAD_ACCUM             = 4             # effective batch = 64
TOTAL_STEPS            = 2000          # raise to 5000 for a longer validation run
WARMUP_STEPS           = 200
EVAL_EVERY             = 100
SAVE_EVERY             = 500
MAX_EVAL_BATCHES       = 40
SESSION_EVAL_CHUNKS    = 8
SESSION_EVAL_BATCH     = 8
CURRICULUM_START_STEP  = 400
CURRICULUM_PROB        = 0.25          # after warmup, fraction of microbatches from memory curriculum
LR_MAX                 = 6e-4
LR_MIN                 = 6e-5
WEIGHT_DECAY           = 0.1
BETAS                  = (0.9, 0.95)
GRAD_CLIP              = 1.0
RESET_MEMORY_EVERY_UPDATES = 2         # optimizer steps per session before reset
COMPILE_MODEL          = False         # can be True later, but start False for clean validation
USE_FULL_12L_768D      = True          # True = architectural validation at declared scale

# -----------------------
# Repro / device / A100 config
# -----------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "CUDA GPU required."
device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
print(f"[GPU] {gpu_name} | VRAM={gpu_props.total_memory/1024**3:.1f} GB")

# A100 optimal settings
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

# Enable PyTorch SDPA fused kernels. On A100 this routes to flash kernels where possible.
try:
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
except Exception:
    pass

bf16_supported = torch.cuda.is_bf16_supported()
assert bf16_supported, "This notebook is designed for bf16-capable GPUs (A100/H100 class)."
amp_dtype = torch.bfloat16
print(f"[AMP] bf16_supported={bf16_supported}; using autocast dtype={amp_dtype}")

# -----------------------
# Clone repo and normalize package layout
# -----------------------
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

def normalize_repo_layout(repo_dir: Path):
    """
    Repo may be flat but imports expect:
      dcortex/
        config.py
        model.py
        memory/*.py
        backbone/*.py
    This function preserves originals and creates the expected package layout.
    """
    flat_files = [repo_dir / x for x in [
        "model.py","config.py","banks.py","query.py","updater.py","readers.py",
        "writer.py","consolidator.py","embeddings.py","transformer.py","fusion_block.py"
    ]]
    if not any(p.exists() for p in flat_files):
        return

    pkg = repo_dir / "dcortex"
    bb = pkg / "backbone"
    mem = pkg / "memory"
    for d in [pkg, bb, mem]:
        d.mkdir(parents=True, exist_ok=True)
        init = d / "__init__.py"
        if not init.exists():
            init.write_text("")

    copy_map = {
        "config.py": pkg / "config.py",
        "model.py": pkg / "model.py",
        "embeddings.py": bb / "embeddings.py",
        "transformer.py": bb / "transformer.py",
        "fusion_block.py": bb / "fusion_block.py",
        "banks.py": mem / "banks.py",
        "query.py": mem / "query.py",
        "updater.py": mem / "updater.py",
        "readers.py": mem / "readers.py",
        "writer.py": mem / "writer.py",
        "consolidator.py": mem / "consolidator.py",
    }
    for src_name, dst in copy_map.items():
        src = repo_dir / src_name
        if src.exists():
            shutil.copyfile(src, dst)

normalize_repo_layout(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

from dcortex.config import DCortexConfig
from dcortex.model import DCortexV2Model
import dcortex.backbone.transformer as tr_mod
import dcortex.backbone.fusion_block as fb_mod
import dcortex.memory.writer as writer_mod

# -----------------------
# Runtime patches:
#   1) self-attn / cross-attn -> SDPA (Flash kernels on A100)
#   2) writer caches gate_probs + route counts for metrics
# -----------------------
def mha_forward_flash(self, h, attention_mask=None):
    B, T, D = h.shape
    qkv = self.qkv(h).view(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]  # [B,H,T,d]
    if attention_mask is not None:
        # attention_mask: [B, T] with 1 = valid, 0 = pad
        # SDPA expects True entries in attn_mask to participate for bool masks.
        # We'll build additive mask instead to stay explicit.
        pad = (attention_mask == 0).view(B, 1, 1, T)
        additive = torch.zeros((B, 1, T, T), device=h.device, dtype=h.dtype)
        additive = additive.masked_fill(pad.expand(B, 1, T, T), torch.finfo(h.dtype).min)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=additive, dropout_p=0.0, is_causal=True)
    else:
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=True)
    out = out.transpose(1, 2).contiguous().view(B, T, D)
    return self.out(out)

def cross_forward_flash(self, h, memory):
    B, T, D = h.shape
    _, K, _ = memory.shape
    q = self.q(h).view(B, T, self.n_heads, self.head_dim).permute(0,2,1,3)
    kv = self.kv(memory).view(B, K, 2, self.n_heads, self.head_dim).permute(2,0,3,1,4)
    k, v = kv[0], kv[1]
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=False)
    out = out.transpose(1,2).contiguous().view(B, T, D)
    return self.out(out)

_orig_writer_forward = writer_mod.MemoryWriter.forward
def writer_forward_with_cache(self, h_pool, updater, banks, step):
    gate_probs = _orig_writer_forward(self, h_pool, updater, banks, step)
    self.last_gate_probs = gate_probs.detach()
    self.last_gate_mean = gate_probs.detach().mean(dim=0)
    choices = gate_probs.detach().argmax(dim=-1)
    counts = torch.bincount(choices, minlength=6).float()
    self.last_choice_counts = counts
    return gate_probs

tr_mod.MultiHeadSelfAttention.forward = mha_forward_flash
fb_mod.CrossAttention.forward = cross_forward_flash
writer_mod.MemoryWriter.forward = writer_forward_with_cache
print("[PATCH] Self-attention and cross-attention now use torch SDPA fused kernels where available.")
print("[PATCH] MemoryWriter now caches gate statistics for metrics.")

# -----------------------
# Tokenizer and memmap pipeline
# -----------------------
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
assert tokenizer.vocab_size == 50257
eot_id = tokenizer.eos_token_id
print(f"[TOKENIZER] vocab={tokenizer.vocab_size}, eos={eot_id}")

train_bin = DATA_DIR / f"tinystories_train_{TRAIN_STORIES}_gpt2.bin"
val_bin   = DATA_DIR / f"tinystories_val_{VAL_STORIES}_gpt2.bin"
meta_json = DATA_DIR / "tinystories_meta.json"

def write_memmap_from_hf(split: str, n_items: int, out_file: Path):
    if out_file.exists():
        print(f"[DATA] Reusing existing memmap: {out_file}")
        return
    ds = load_dataset("roneneldan/TinyStories", split=split)
    n_items = min(n_items, len(ds))
    # First pass: count tokens.
    total_tokens = 0
    for i in tqdm(range(n_items), desc=f"[{split}] counting tokens"):
        ids = tokenizer.encode(ds[i]["text"], add_special_tokens=False)
        total_tokens += len(ids) + 1  # + eos
    arr = np.memmap(out_file, dtype=np.uint16, mode="w+", shape=(total_tokens,))
    w = 0
    doc_offsets = []
    for i in tqdm(range(n_items), desc=f"[{split}] tokenizing"):
        ids = tokenizer.encode(ds[i]["text"], add_special_tokens=False)
        ids.append(eot_id)
        doc_offsets.append(int(w))
        arr[w:w+len(ids)] = np.asarray(ids, dtype=np.uint16)
        w += len(ids)
    arr.flush()
    meta = {}
    if meta_json.exists():
        meta = json.loads(meta_json.read_text())
    meta[split] = {
        "items": n_items,
        "tokens": total_tokens,
        "bin": str(out_file),
        "doc_offsets_head": doc_offsets[:1000],
    }
    meta_json.write_text(json.dumps(meta, indent=2))
    print(f"[DATA] Wrote {out_file} | tokens={total_tokens:,}")

write_memmap_from_hf("train", TRAIN_STORIES, train_bin)
write_memmap_from_hf("validation", VAL_STORIES, val_bin)

train_tokens = np.memmap(train_bin, dtype=np.uint16, mode="r")
val_tokens   = np.memmap(val_bin, dtype=np.uint16, mode="r")
print(f"[DATA] train_tokens={len(train_tokens):,} | val_tokens={len(val_tokens):,}")

def get_random_batch(tokens_memmap, batch_size: int, seq_len: int, device: torch.device):
    max_idx = len(tokens_memmap) - seq_len - 1
    ix = np.random.randint(0, max_idx, size=(batch_size,))
    x = np.stack([tokens_memmap[i:i+seq_len].astype(np.int64) for i in ix])
    y = np.stack([tokens_memmap[i+1:i+1+seq_len].astype(np.int64) for i in ix])
    x = torch.from_numpy(x).to(device=device, dtype=torch.long, non_blocking=True)
    y = torch.from_numpy(y).to(device=device, dtype=torch.long, non_blocking=True)
    return x, y

# -----------------------
# Synthetic memory curriculum
# -----------------------
NAMES  = ["Ava","Liam","Milo","Zara","Nina","Owen","Iris","Noah","Luna","Aria","Ezra","Sage"]
COLORS = ["red","blue","green","yellow","purple","orange","silver","white","black","gold"]
PLACES = ["kitchen","garden","attic","library","station","tower","harbor","workshop","forest","cellar"]
ITEMS  = ["key","book","lamp","coin","map","ring","cloak","sword","box","vase"]

def make_memory_text():
    # Build a short fact/update/conflict/retrieval narrative with delayed recall.
    n1, n2 = random.sample(NAMES, 2)
    c1, c2 = random.sample(COLORS, 2)
    p1, p2 = random.sample(PLACES, 2)
    it1, it2 = random.sample(ITEMS, 2)
    fillers = [
        f"{n1} walked through the {p1} and looked at the old {it1}.",
        f"{n2} stayed near the {p2} and carried a {it2}.",
        f"The story paused while the wind moved through the hall.",
        f"Nobody answered for a while and the room stayed quiet.",
    ]
    random.shuffle(fillers)
    prompt = (
        f"Remember these facts. {n1}'s color is {c1}. {n1} keeps the {it1} in the {p1}. "
        f"{n2}'s color is {c2}. {n2} keeps the {it2} in the {p2}. "
        + " ".join(fillers) + " "
        f"Update: {n1}'s color is now {c2}. "
        f"Question: what is {n1}'s color now? Answer: {c2}. "
        f"Question: where does {n2} keep the {it2}? Answer: {p2}."
    )
    return prompt

def build_curriculum_batch(batch_size: int, seq_len: int, device: torch.device):
    texts = [make_memory_text() for _ in range(batch_size)]
    ids_list = []
    for t in texts:
        ids = tokenizer.encode(t, add_special_tokens=False) + [eot_id]
        if len(ids) < seq_len + 1:
            reps = ((seq_len + 1) // len(ids)) + 1
            ids = (ids * reps)[:seq_len + 1]
        else:
            start = random.randint(0, max(0, len(ids) - (seq_len + 1)))
            ids = ids[start:start+seq_len+1]
        ids_list.append(ids)
    arr = np.asarray(ids_list, dtype=np.int64)
    x = torch.from_numpy(arr[:, :-1]).to(device=device, dtype=torch.long)
    y = torch.from_numpy(arr[:, 1:]).to(device=device, dtype=torch.long)
    return x, y, texts

# -----------------------
# Model / optimizer / scheduler
# -----------------------
cfg = DCortexConfig()
if not USE_FULL_12L_768D:
    cfg = cfg.small_test()
    cfg.vocab_size = 50257
    cfg.max_seq_len = min(cfg.max_seq_len, SEQ_LEN)
else:
    cfg.max_seq_len = SEQ_LEN

model = DCortexV2Model(cfg).to(device)

# Optional compile after patching. Keep disabled by default for debugging clarity.
if COMPILE_MODEL and hasattr(torch, "compile"):
    model = torch.compile(model)

# Parameter groups: decay on weights, not on norms/biases/embeddings ties
decay, no_decay = [], []
for n, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if p.dim() >= 2 and "norm" not in n.lower() and not n.endswith(".bias"):
        decay.append(p)
    else:
        no_decay.append(p)

optimizer = AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=LR_MAX, betas=BETAS, fused=True if "fused" in AdamW.__init__.__code__.co_varnames else False
)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return float(step + 1) / float(max(1, WARMUP_STEPS))
    progress = (step - WARMUP_STEPS) / float(max(1, TOTAL_STEPS - WARMUP_STEPS))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return (LR_MIN / LR_MAX) + (1 - (LR_MIN / LR_MAX)) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

# -----------------------
# Helpers
# -----------------------
@torch.no_grad()
def memory_metrics_snapshot(model):
    snap = model.memory_snapshot()
    occ = {k: int(v["occupied"]) for k, v in snap.items()}
    cap = {k: int(v["capacity"]) for k, v in snap.items()}
    util = {k: (occ[k] / max(1, cap[k])) for k in occ}
    return {"occupied": occ, "capacity": cap, "utilization": util}

def gate_metrics(writer):
    if not hasattr(writer, "last_gate_probs"):
        return {
            "entropy": float("nan"),
            "route_mean": [float("nan")] * 6,
            "choice_counts": [0.0] * 6
        }
    p = writer.last_gate_probs
    mean_p = p.mean(dim=0)
    entropy = float((-(p.clamp_min(1e-9) * p.clamp_min(1e-9).log()).sum(dim=-1)).mean().item())
    return {
        "entropy": entropy,
        "route_mean": [float(x) for x in mean_p.detach().cpu()],
        "choice_counts": [float(x) for x in writer.last_choice_counts.detach().cpu()],
    }

def compute_loss(logits, targets):
    B, T, V = logits.shape
    return F.cross_entropy(logits.view(B*T, V), targets.view(B*T))

def atomic_torch_save(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(delete=False, dir=str(path.parent), suffix=".tmp") as tmp:
        tmp_path = Path(tmp.name)
    try:
        torch.save(obj, tmp_path)
        os.replace(tmp_path, path)
    finally:
        if tmp_path.exists():
            tmp_path.unlink(missing_ok=True)

@torch.no_grad()
def run_eval(model, tokens_memmap, n_batches=MAX_EVAL_BATCHES, seq_len=SEQ_LEN, batch_size=MICRO_BSZ):
    model.eval()
    losses = []
    gate_ent = []
    util_hist = []
    model.reset_memory()
    for _ in range(n_batches):
        x, y = get_random_batch(tokens_memmap, batch_size, seq_len, device)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits = model(x, write_memory=False)
            loss = compute_loss(logits, y)
        losses.append(float(loss.item()))
        gm = gate_metrics(model.writer)
        if not math.isnan(gm["entropy"]):
            gate_ent.append(gm["entropy"])
        util_hist.append(memory_metrics_snapshot(model)["utilization"])
    ppl = math.exp(sum(losses)/len(losses))
    # average bank utils
    banks = util_hist[0].keys()
    util_mean = {k: float(np.mean([u[k] for u in util_hist])) for k in banks}
    return {
        "loss": float(np.mean(losses)),
        "ppl": float(ppl),
        "gate_entropy": float(np.mean(gate_ent)) if gate_ent else float("nan"),
        "utilization": util_mean,
    }

@torch.no_grad()
def run_session_memory_eval(model, tokens_memmap, batch_size=SESSION_EVAL_BATCH, seq_len=SEQ_LEN, chunks=SESSION_EVAL_CHUNKS):
    """
    Compare sequential chunks with persistent memory vs resetting memory each chunk.
    This gives a stronger signal than plain random-perplexity.
    """
    model.eval()
    max_idx = len(tokens_memmap) - (chunks + 1) * seq_len - 1
    starts = np.random.randint(0, max_idx, size=(batch_size,))
    session_losses_mem = []
    session_losses_reset = []

    # persistent-memory pass
    model.reset_memory()
    for c in range(chunks):
        x = np.stack([tokens_memmap[s + c*seq_len : s + (c+1)*seq_len].astype(np.int64) for s in starts])
        y = np.stack([tokens_memmap[s + c*seq_len + 1 : s + (c+1)*seq_len + 1].astype(np.int64) for s in starts])
        x = torch.from_numpy(x).to(device=device, dtype=torch.long)
        y = torch.from_numpy(y).to(device=device, dtype=torch.long)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits = model(x, write_memory=True)
            loss = compute_loss(logits, y)
        session_losses_mem.append(float(loss.item()))

    # reset-every-chunk pass
    for c in range(chunks):
        model.reset_memory()
        x = np.stack([tokens_memmap[s + c*seq_len : s + (c+1)*seq_len].astype(np.int64) for s in starts])
        y = np.stack([tokens_memmap[s + c*seq_len + 1 : s + (c+1)*seq_len + 1].astype(np.int64) for s in starts])
        x = torch.from_numpy(x).to(device=device, dtype=torch.long)
        y = torch.from_numpy(y).to(device=device, dtype=torch.long)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits = model(x, write_memory=True)
            loss = compute_loss(logits, y)
        session_losses_reset.append(float(loss.item()))

    gain = float(np.mean(session_losses_reset) - np.mean(session_losses_mem))
    return {
        "persistent_loss_curve": session_losses_mem,
        "reset_loss_curve": session_losses_reset,
        "memory_gain_loss_delta": gain,  # positive is better
    }

def save_checkpoint(step, history):
    ckpt = {
        "step": step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "config": asdict(cfg),
        "history": history,
        "rng": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all(),
        },
    }
    path = CKPT_DIR / f"step_{step:07d}.pt"
    atomic_torch_save(ckpt, path)
    print(f"[CKPT] Saved {path}")

# -----------------------
# Pre-flight validation
# -----------------------
model.train()
model.reset_memory()
x0, y0 = get_random_batch(train_tokens, batch_size=2, seq_len=min(64, SEQ_LEN), device=device)
with torch.autocast(device_type="cuda", dtype=amp_dtype):
    logits0 = model(x0, write_memory=True)
    loss0 = compute_loss(logits0, y0)
loss0.backward()
grad_norm0 = float(torch.nn.utils.clip_grad_norm_(model.parameters(), 1e9).item())
optimizer.zero_grad(set_to_none=True)
print(f"[SMOKE] initial loss={loss0.item():.4f} | grad_norm={grad_norm0:.4f} | snapshot={memory_metrics_snapshot(model)}")

# -----------------------
# Training loop
# -----------------------
history = {
    "step": [],
    "train_loss": [],
    "real_loss": [],
    "curriculum_loss": [],
    "lr": [],
    "grad_norm": [],
    "gate_entropy": [],
    "route_mean": [],
    "bank_utilization": [],
    "eval": [],
    "session_eval": [],
    "tokens_seen": [],
    "throughput_tok_s": [],
}

tokens_per_step = MICRO_BSZ * GRAD_ACCUM * SEQ_LEN
start_time = time.time()
optimizer_step = 0
real_loss_ema = None
curr_loss_ema = None

model.train()
optimizer.zero_grad(set_to_none=True)
model.reset_memory()

pbar = tqdm(range(1, TOTAL_STEPS + 1), desc="training")
for step in pbar:
    t0 = time.time()
    accum_loss = 0.0
    micro_real = []
    micro_curr = []

    for micro in range(GRAD_ACCUM):
        use_curriculum = (step >= CURRICULUM_START_STEP) and (random.random() < CURRICULUM_PROB)
        if use_curriculum:
            x, y, _texts = build_curriculum_batch(MICRO_BSZ, SEQ_LEN, device)
        else:
            x, y = get_random_batch(train_tokens, MICRO_BSZ, SEQ_LEN, device)

        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits = model(x, write_memory=True)
            loss_main = compute_loss(logits, y)

            # Auxiliary loss 1: route entropy penalty (encourage decisive writing, not uniform routing)
            if hasattr(model.writer, "last_gate_probs"):
                p = model.writer.last_gate_probs
                gate_entropy = (-(p.clamp_min(1e-9) * p.clamp_min(1e-9).log()).sum(dim=-1)).mean()
                loss_gate_entropy = 0.01 * gate_entropy
            else:
                loss_gate_entropy = torch.tensor(0.0, device=device)

            # Auxiliary loss 2: discourage collapse to skip-only routing
            if hasattr(model.writer, "last_gate_probs"):
                mean_p = model.writer.last_gate_probs.mean(dim=0)
                skip_prob = mean_p[-1]
                loss_skip = 0.02 * skip_prob
            else:
                loss_skip = torch.tensor(0.0, device=device)

            loss = (loss_main + loss_gate_entropy + loss_skip) / GRAD_ACCUM

        loss.backward()
        accum_loss += float(loss_main.item())

        if use_curriculum:
            micro_curr.append(float(loss_main.item()))
        else:
            micro_real.append(float(loss_main.item()))

    grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP).item())
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    optimizer_step += 1

    # session boundary
    if optimizer_step % RESET_MEMORY_EVERY_UPDATES == 0:
        model.reset_memory()

    dt = time.time() - t0
    tok_s = tokens_per_step / max(dt, 1e-6)
    train_loss = accum_loss / GRAD_ACCUM
    if micro_real:
        real_loss = float(np.mean(micro_real))
        real_loss_ema = real_loss if real_loss_ema is None else (0.95 * real_loss_ema + 0.05 * real_loss)
    else:
        real_loss = float("nan")
    if micro_curr:
        curr_loss = float(np.mean(micro_curr))
        curr_loss_ema = curr_loss if curr_loss_ema is None else (0.95 * curr_loss_ema + 0.05 * curr_loss)
    else:
        curr_loss = float("nan")

    gm = gate_metrics(model.writer)
    mm = memory_metrics_snapshot(model)

    history["step"].append(step)
    history["train_loss"].append(train_loss)
    history["real_loss"].append(real_loss)
    history["curriculum_loss"].append(curr_loss)
    history["lr"].append(float(optimizer.param_groups[0]["lr"]))
    history["grad_norm"].append(grad_norm)
    history["gate_entropy"].append(gm["entropy"])
    history["route_mean"].append(gm["route_mean"])
    history["bank_utilization"].append(mm["utilization"])
    history["tokens_seen"].append(step * tokens_per_step)
    history["throughput_tok_s"].append(tok_s)

    pbar.set_postfix({
        "loss": f"{train_loss:.3f}",
        "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
        "gnorm": f"{grad_norm:.2f}",
        "tok/s": f"{tok_s/1e3:.0f}k",
        "gateH": f"{gm['entropy']:.2f}",
        "state": f"{mm['utilization']['state']:.2f}",
        "work": f"{mm['utilization']['working']:.2f}",
    })

    if step % EVAL_EVERY == 0 or step == 1 or step == TOTAL_STEPS:
        eval_stats = run_eval(model, val_tokens, n_batches=MAX_EVAL_BATCHES)
        sess_stats = run_session_memory_eval(model, val_tokens)
        history["eval"].append({"step": step, **eval_stats})
        history["session_eval"].append({"step": step, **sess_stats})
        print(f"\n[EVAL step={step}] loss={eval_stats['loss']:.4f} ppl={eval_stats['ppl']:.2f} "
              f"gateH={eval_stats['gate_entropy']:.3f} util={eval_stats['utilization']} "
              f"| session memory gain={sess_stats['memory_gain_loss_delta']:.4f}")
        model.train()

    if step % SAVE_EVERY == 0 or step == TOTAL_STEPS:
        save_checkpoint(step, history)

# -----------------------
# Final comprehensive report
# -----------------------
elapsed = time.time() - start_time
final_eval = history["eval"][-1]
final_session = history["session_eval"][-1]

summary = {
    "run_name": RUN_NAME,
    "gpu": gpu_name,
    "bf16": bf16_supported,
    "flash_sdpa_enabled": True,
    "seq_len": SEQ_LEN,
    "micro_bsz": MICRO_BSZ,
    "grad_accum": GRAD_ACCUM,
    "total_steps": TOTAL_STEPS,
    "tokens_seen": history["tokens_seen"][-1],
    "elapsed_sec": elapsed,
    "avg_tok_s_last100": float(np.mean(history["throughput_tok_s"][-100:])),
    "final_train_loss": float(history["train_loss"][-1]),
    "final_val_loss": final_eval["loss"],
    "final_val_ppl": final_eval["ppl"],
    "final_session_memory_gain": final_session["memory_gain_loss_delta"],
    "final_bank_utilization": history["bank_utilization"][-1],
    "final_route_mean": history["route_mean"][-1],
    "notes": [
        "No GradScaler used by design: A100 bf16 path.",
        "Attention patched to SDPA to activate fused flash kernels on supported shapes.",
        "Repo package layout normalized at runtime if the source is flat.",
        "This validates end-to-end trainability, gradient health, memory utilization, and memory-vs-reset session behavior."
    ]
}
(REPORT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
(REPORT_DIR / "history.json").write_text(json.dumps(history, indent=2))

# -----------------------
# Plots
# -----------------------
steps = np.array(history["step"])
fig = plt.figure(figsize=(18, 16))

ax = plt.subplot(3,2,1)
ax.plot(steps, history["train_loss"], label="train")
real = np.array(history["real_loss"], dtype=np.float64)
curr = np.array(history["curriculum_loss"], dtype=np.float64)
if np.isfinite(real).any(): ax.plot(steps[np.isfinite(real)], real[np.isfinite(real)], label="real-text", alpha=0.8)
if np.isfinite(curr).any(): ax.plot(steps[np.isfinite(curr)], curr[np.isfinite(curr)], label="curriculum", alpha=0.8)
ax.set_title("Training losses")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend()

ax = plt.subplot(3,2,2)
ax.plot(steps, history["lr"], label="lr")
ax.set_title("Learning rate")
ax.set_xlabel("step"); ax.set_ylabel("lr")

ax = plt.subplot(3,2,3)
ax.plot(steps, history["grad_norm"], label="grad_norm")
ax.plot(steps, history["gate_entropy"], label="gate_entropy")
ax.set_title("Gradient norm and writer entropy")
ax.set_xlabel("step"); ax.legend()

ax = plt.subplot(3,2,4)
bank_names = list(history["bank_utilization"][-1].keys())
for b in bank_names:
    ax.plot(steps, [x[b] for x in history["bank_utilization"]], label=b)
ax.set_title("Memory bank utilization")
ax.set_xlabel("step"); ax.set_ylabel("occupancy ratio"); ax.legend()

ax = plt.subplot(3,2,5)
route_arr = np.array(history["route_mean"])
route_names = ["state","episode_obj","conflict","archive","working","skip"]
for i, name in enumerate(route_names):
    ax.plot(steps, route_arr[:, i], label=name)
ax.set_title("Writer route probabilities")
ax.set_xlabel("step"); ax.legend(ncol=2)

ax = plt.subplot(3,2,6)
eval_steps = [x["step"] for x in history["eval"]]
eval_ppl = [x["ppl"] for x in history["eval"]]
sess_gain = [x["memory_gain_loss_delta"] for x in history["session_eval"]]
ax.plot(eval_steps, eval_ppl, label="val_ppl")
ax2 = ax.twinx()
ax2.plot(eval_steps, sess_gain, label="session_memory_gain", linestyle="--")
ax.set_title("Validation perplexity and memory gain")
ax.set_xlabel("step")
ax.set_ylabel("ppl")
ax2.set_ylabel("loss delta (reset - persistent)")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="best")

plt.tight_layout()
plot_path = REPORT_DIR / "training_report.png"
plt.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n" + "="*90)
print("COMPREHENSIVE VALIDATION SUMMARY")
print("="*90)
for k, v in summary.items():
    print(f"{k}: {v}")
print("="*90)
print(f"Artifacts written to: {REPORT_DIR}")
print(f"Checkpoints written to: {CKPT_DIR}")
